# Hoja de trabajo 2

## Task 2 y 3

### Integrantes
* Sergio Orellana 221122
* Rodrigo Mansilla 22611
* Ricardo Chuy 221007

## Simulador de *k-armed bandit*

El notebook implementa el entorno, las estrategias, los experimentos y las visualizaciones. Los comentarios breves indican la función de cada parte.

In [ ]:
# NumPy permite trabajar con arreglos y generar valores aleatorios.
import numpy as np

# Matplotlib se utilizará para construir las cuatro gráficas solicitadas.
import matplotlib.pyplot as plt

# Facilita crear copias independientes de una estrategia para cada episodio.
from copy import deepcopy

### 1. Entorno

La clase representa las `k` acciones, sus valores reales ocultos y la recompensa obtenida. En modo no estacionario debe modificar esos valores cada cierto número de pasos.

In [ ]:
class KArmedBandit:
    def __init__(self, k=10, stationary=True, perturb_every=100, perturb_scale=0.01, seed=None):
        # Guarda la configuración y crea un generador aleatorio reproducible.
        self.k = k
        self.stationary = stationary
        self.perturb_every = perturb_every
        self.perturb_scale = perturb_scale
        self.rng = np.random.default_rng(seed)
        self.q_true = None
        self.step_count = 0
        self.reset()

    def reset(self):
        # Crea un valor real oculto para cada acción y reinicia el tiempo.
        self.q_true = self.rng.normal(0.0, 1.0, size=self.k)
        self.step_count = 0
        return self.q_true.copy()

    def step(self, action):
        # Verifica que la acción corresponda a uno de los brazos disponibles.
        if not 0 <= action < self.k:
            raise ValueError(f'La acción debe estar entre 0 y {self.k - 1}.')

        # La recompensa observada contiene ruido alrededor del valor real.
        reward = self.rng.normal(self.q_true[action], 1.0)
        self.step_count += 1

        # En el caso no estacionario, todos los valores realizan un pequeño cambio.
        if (not self.stationary and self.perturb_every > 0
                and self.step_count % self.perturb_every == 0):
            self.q_true += self.rng.normal(0.0, self.perturb_scale, size=self.k)

        return reward

    def optimal_value(self):
        # Devuelve el valor real de la mejor acción en el estado actual.
        return float(np.max(self.q_true))

### 2. Estrategias

Cada estrategia debe elegir una acción con `select_action` y actualizar sus estimaciones con `update`. Usa `step_size=None` para paso variable y un número fijo para paso constante.

In [ ]:
class BanditStrategy:
    def __init__(self, k, step_size=None, seed=None):
        # q_estimate almacena estimaciones; action_counts registra selecciones.
        self.k = k
        self.step_size = step_size
        self.rng = np.random.default_rng(seed)
        self.q_estimate = np.zeros(k)
        self.action_counts = np.zeros(k, dtype=int)
        self.total_steps = 0

    def reset(self):
        # Reinicia el aprendizaje antes de comenzar un episodio independiente.
        self.q_estimate.fill(0.0)
        self.action_counts.fill(0)
        self.total_steps = 0

    def select_action(self):
        # Cada subclase implementará aquí su criterio de selección.
        raise NotImplementedError

    def update(self, action, reward):
        # Registra la visita antes de calcular el tamaño del paso variable.
        self.action_counts[action] += 1
        self.total_steps += 1

        # Sin valor fijo se usa el promedio muestral; con valor, un paso constante.
        alpha = (1.0 / self.action_counts[action]
                 if self.step_size is None else self.step_size)
        self.q_estimate[action] += alpha * (reward - self.q_estimate[action])

In [ ]:
class Greedy(BanditStrategy):
    def select_action(self):
        # Elige al azar entre las acciones empatadas con la mayor estimación.
        best_actions = np.flatnonzero(self.q_estimate == self.q_estimate.max())
        return int(self.rng.choice(best_actions))


class EpsilonGreedy(BanditStrategy):
    def __init__(self, k, epsilon=0.1, step_size=None, seed=None):
        super().__init__(k, step_size, seed)
        # epsilon controla con qué frecuencia se explora.
        self.epsilon = epsilon

    def select_action(self):
        # Explora una acción cualquiera con probabilidad epsilon.
        if self.rng.random() < self.epsilon:
            return int(self.rng.integers(self.k))

        # En caso contrario, explota una de las mejores estimaciones.
        best_actions = np.flatnonzero(self.q_estimate == self.q_estimate.max())
        return int(self.rng.choice(best_actions))


class UCB1(BanditStrategy):
    def __init__(self, k, c=2.0, step_size=None, seed=None):
        super().__init__(k, step_size, seed)
        # c controla cuánto favorece la estrategia a acciones poco probadas.
        self.c = c

    def select_action(self):
        # Primero garantiza que cada acción se haya probado al menos una vez.
        untried = np.flatnonzero(self.action_counts == 0)
        if len(untried) > 0:
            return int(self.rng.choice(untried))

        # Combina el valor estimado con una bonificación por incertidumbre.
        bonus = self.c * np.sqrt(np.log(self.total_steps) / self.action_counts)
        scores = self.q_estimate + bonus
        best_actions = np.flatnonzero(scores == scores.max())
        return int(self.rng.choice(best_actions))

### 3. Experimentos

Cada episodio debe crear un entorno y una estrategia nuevos. Los resultados se promedian por paso; el *regret* se acumula dentro de cada episodio antes de promediarlo.

In [ ]:
N_EPISODES = 500
N_STEPS = 1000
K = 10


def run_experiment(strategy_template, stationary, n_episodes=N_EPISODES, n_steps=N_STEPS):
    # Estas matrices guardan una trayectoria por episodio.
    rewards = np.zeros((n_episodes, n_steps))
    regrets = np.zeros((n_episodes, n_steps))

    for episode in range(n_episodes):
        # Cada episodio usa otro entorno, pero una semilla reproducible.
        bandit = KArmedBandit(
            k=strategy_template.k, stationary=stationary, seed=episode
        )

        # La copia evita compartir aprendizaje entre episodios.
        strategy = deepcopy(strategy_template)
        strategy.rng = np.random.default_rng(100_000 + episode)
        strategy.reset()

        for step in range(n_steps):
            # Se mide el valor elegido antes de que el entorno pueda perturbarse.
            action = strategy.select_action()
            best_value = bandit.optimal_value()
            chosen_value = bandit.q_true[action]

            # Ejecuta la acción, aprende de la recompensa y registra ambas métricas.
            reward = bandit.step(action)
            strategy.update(action, reward)
            rewards[episode, step] = reward
            regrets[episode, step] = best_value - chosen_value

    # Promedia episodios; el regret se acumula a lo largo de los pasos.
    mean_reward = rewards.mean(axis=0)
    mean_cumulative_regret = np.cumsum(regrets, axis=1).mean(axis=0)
    return mean_reward, mean_cumulative_regret

In [ ]:
# Estos hiperparámetros pueden modificarse para estudiar su efecto.
strategies = {
    'Greedy': Greedy(K, step_size=None),
    'Epsilon-greedy': EpsilonGreedy(K, epsilon=0.1, step_size=None),
    'UCB1': UCB1(K, c=2.0, step_size=None),
}

# Separa resultados estacionarios y no estacionarios.
results = {'stationary': {}, 'non_stationary': {}}

# Ejecuta cada estrategia en ambos tipos de entorno.
for name, strategy in strategies.items():
    results['stationary'][name] = run_experiment(strategy, stationary=True)
    results['non_stationary'][name] = run_experiment(strategy, stationary=False)

### 4. Visualización

La cuadrícula contiene exactamente las cuatro comparaciones pedidas. Cada eje debe mostrar una curva por estrategia.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))

# Fila superior: entorno estacionario. Fila inferior: no estacionario.
plot_specs = [
    (axes[0, 0], 'stationary', 0, 'Recompensa promedio — estacionario', 'Recompensa'),
    (axes[0, 1], 'stationary', 1, 'Regret acumulado — estacionario', 'Regret'),
    (axes[1, 0], 'non_stationary', 0, 'Recompensa promedio — no estacionario', 'Recompensa'),
    (axes[1, 1], 'non_stationary', 1, 'Regret acumulado — no estacionario', 'Regret'),
]

for ax, environment, metric_index, title, ylabel in plot_specs:
    # Dibuja la métrica correspondiente para cada estrategia.
    for strategy_name, metrics in results[environment].items():
        ax.plot(metrics[metric_index], label=strategy_name)
    ax.set_title(title)
    ax.set_xlabel('Paso')
    ax.set_ylabel(ylabel)
    ax.grid(alpha=0.3)
    ax.legend()

plt.tight_layout()
plt.show()

### Lista de comprobación

- [ ] El entorno funciona en modo estacionario y no estacionario.
- [ ] La perturbación ocurre con la frecuencia configurada.
- [ ] Las tres estrategias implementan `select_action` y `update`.
- [ ] La actualización acepta paso variable y paso constante.
- [ ] Se ejecutan al menos 500 episodios de 1000 pasos.
- [ ] Se calculan recompensa promedio por paso y regret acumulado.
- [ ] Se generan las cuatro gráficas solicitadas.